# Como a intermitência da geração eólica/solar nos submercados afeta a volatilidade do PLD, e é possível antecipar janelas de risco de preço para apoiar decisões de gestão de portfólio de energia?![](path)

tabelas:
  - bronze.ons_balanco_energia_subsistema   (2000–2026, horário)
  - bronze.ccee_pld_horario                 (2021–2026, horário)
  - bronze.ccee_pld_historico_semanal       (2001–2020, semanal, PLD não era horário)

# SETUP

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import requests, io

CATALOG = "portfolio_energia"
SCHEMA_BRONZE = "bronze"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")

def forcar_schema_string(sdf, exclude_cols=("_ano_arquivo", "_source_file", "_source", "_ingestion_timestamp")):
    "forças todas as colunas para string para evitar erros de schema"
    for c in sdf.columns:
        if c not in exclude_cols:
            sdf = sdf.withColumn(c, F.col(c).cast("string"))
    return sdf

def registros_para_spark(registros):
    """Converte lista de dicts (vinda de API JSON) pra Spark via pandas,
    evitando conflito de tipos LongType/DoubleType na inferência linha a linha."""
    df_pd = pd.DataFrame(registros)
    return spark.createDataFrame(df_pd)

# INGESTÃO DA ONS

In [0]:
BASE_URL_ONS = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/balanco_energia_subsistema_ho/BALANCO_ENERGIA_SUBSISTEMA_{ano}.parquet"

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; PortfolioEnergiaBot/1.0)"})

anos = range(2000, 2026)
dfs_spark = []
falhas = []

for ano in anos:
    url = BASE_URL_ONS.format(ano=ano)
    try:
        resp = session.get(url, timeout=120)
        resp.raise_for_status()
        df_pd = pd.read_parquet(io.BytesIO(resp.content))
        df_pd = df_pd.replace('', None)
        df_pd["_ano_arquivo"] = ano
        df_pd["_source_file"] = url
        sdf = spark.createDataFrame(df_pd)
        sdf = forcar_schema_string(sdf)
        dfs_spark.append(sdf)
        print(f"OK  {ano}: {sdf.count()} linhas")
        del df_pd
    except Exception as e:
        falhas.append((ano, str(e)))
        print(f"FALHOU {ano}: {e}")

df_ons = dfs_spark[0]
for d in dfs_spark[1:]:
    df_ons = df_ons.unionByName(d, allowMissingColumns=True)

df_ons = df_ons.withColumn("_ingestion_timestamp", F.current_timestamp())

(
    df_ons.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ano_arquivo")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.ons_balanco_energia_subsistema")
)

print(f"\nTotal carregado: {df_ons.count()} linhas | Falhas: {falhas}")

# INGESTÃO DADOS HORÁRIOS DA CEE

In [0]:
from curl_cffi import requests as creq
import time

BASE_API_CCEE = "https://dadosabertos.ccee.org.br/api/3/action/datastore_search"

resource_ids_horario = {
    2021: "51922462-16b4-4c64-8327-4e14d6ee8c6c",
    2022: "723cf7e6-6c29-4da6-aa39-e4c8804baf65",
    2023: "5fc317af-7191-4f8a-94e7-f77c56c747b3",
    2024: "1b5b6946-8036-4622-a7a3-b21f33fc52b7",
    2025: "2a180a6b-f092-43eb-9f82-a48798b803dc",
    2026: "3f279d6b-1069-42f7-9b0a-217b084729c4",
}

def baixar_resource_ckan_curl(resource_id, page_size=32000):
    """Pagina um resource do CKAN via curl_cffi com impersonation de Chrome
    (necessário para contornar bloqueio de TLS fingerprint do WAF da CCEE)."""
    registros = []
    offset = 0
    while True:
        params = {"resource_id": resource_id, "limit": page_size, "offset": offset}
        resp = creq.get(BASE_API_CCEE, params=params, impersonate="chrome", timeout=60)
        resp.raise_for_status()
        result = resp.json()["result"]
        batch = result["records"]
        if not batch:
            break
        registros.extend(batch)
        offset += page_size
        if len(batch) < page_size:
            break
        time.sleep(0.2)
    return registros

todos_registros = []
falhas_ccee = []
for ano, rid in resource_ids_horario.items():
    try:
        regs = baixar_resource_ckan_curl(rid)
        for r in regs:
            r["_ano_arquivo"] = ano
        todos_registros.extend(regs)
        print(f"OK  {ano}: {len(regs)} registros")
    except Exception as e:
        falhas_ccee.append((ano, str(e)))
        print(f"FALHOU {ano}: {e}")

df_pld_horario = registros_para_spark(todos_registros)
df_pld_horario = (
    df_pld_horario
    .withColumn("_source", F.lit("ccee_datastore_api"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)
df_pld_horario = forcar_schema_string(df_pld_horario)

(
    df_pld_horario.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ano_arquivo")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.ccee_pld_horario")
)

print(f"\nTotal PLD horário: {df_pld_horario.count()} linhas | Falhas: {falhas_ccee}")

# INGESTÃO DADOS SEMANAIS DA CCEE (formato antigo do PLD)

In [0]:
RESOURCE_ID_SEMANAL = "7c90a379-5e98-46ff-a11b-9120bcf81ac4"

registros_semanal = baixar_resource_ckan_curl(RESOURCE_ID_SEMANAL)

df_pld_semanal = registros_para_spark(registros_semanal)
df_pld_semanal = (
    df_pld_semanal
    .withColumn("_source", F.lit("ccee_datastore_api"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)
df_pld_semanal = forcar_schema_string(df_pld_semanal)

(
    df_pld_semanal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.ccee_pld_historico_semanal")
)

print(f"Total PLD semanal histórico: {df_pld_semanal.count()} linhas")